### Load data

In [1]:
import random
import pandas as pd

In [2]:
df = pd.read_csv("amazon_beauty_products_1000.csv")

In [3]:
df.head()

,parent_asin,title,content,store,price,average_rating
0,B01LZG89RR,Careline Super Long Lasting Automatic Eye Penc...,eye automatic pencil,Careline,NaN,4.7
1,B09LLV9TPV,Dailinna Corn Wave Ponytail Extension Clip in ...,Style: Drawstring corn wave ponytail hair exte...,Dailinna,NaN,3.0
2,B07J4VNYW1,"ZUM Holiday Mint Blitzum Mist, 4 FZ",Spray a little merry wherever and whenever a m...,Indigo Wild,NaN,4.5
3,B08KFFB2PP,Perfectly Posh PLAY DATE Super Moisturizing Nu...,Super Moisturizing Nutrient-Rich Coconut Oil i...,Perfectly Posh,NaN,3.9
4,B09MQ53X93,New for 2021 Bath and Body Works Gingham Heart...,All items come from a smoke and pet free envir...,Bath & Body Works,38.9,4.7


### Setup llm client

In [4]:
import openai, instructor
from pydantic import BaseModel
from typing import List

In [5]:
client = instructor.from_openai(openai.OpenAI())

In [6]:
class Questions(BaseModel):
    question: str

In [7]:
class Product(BaseModel):
    parent_asin: str
    title: str
    content: str

In [13]:
products = []
for _, row in df.iterrows():
  products.append(Product(
      parent_asin=row['parent_asin'],
      title=row['title'],
      content=row['content']
  ))

In [9]:
extra_rules = [
    "Queries can have minor spelling errors",
    "Mix and match key product attributes (ingredients, benefits, brand, year, features, etc)"
]

In [10]:
prompt = """
Generate a hypothetical query / keywords that a user would search on Amazon to get back this product

Product title:
{{ title }}

Examples of good search queries:
- "ponytail clip"
- "mint spray"
- "automatic eyeliner"
- "Massage Chair By OOTORI"

Rules
- Generate a search query, NOT a question
- Use 2-5 keywords that customers would actually type
- Keep some of them vague and others specific
- {{ extra_rule }}
"""

In [17]:
def get_response(title):
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        response_model=Questions,
        context={
        "title": title,
            "extra_rule": random.choice(extra_rules)
        }, 
    )
    return resp.question

In [24]:
asins = [p.parent_asin for p in products]

In [25]:
queries = [get_response(p.title) for p in products]

In [26]:
query_df =pd.DataFrame({
    "parent_asin": asins,
    "query": queries
})

In [27]:
query_df.head()

,parent_asin,query
0,B01LZG89RR,Careline automatic eyeliner shiny blue pencil
1,B09LLV9TPV,Dailinna Corn Wave Ponytail Extension Clip in ...
2,B07J4VNYW1,ZUM Holiday Mint Blitzum Mist 4 FZ
3,B08KFFB2PP,coconut oil moisturizer
4,B09MQ53X93,New for 2021 Bath and Body Works Gingham Heart...
